# `latent_pair_codec` Colab Runner

Use this notebook from VS Code with the Google Colab extension attached to a **GPU** runtime.

Recommended GPU priority:
1. `A100`
2. `L4`
3. `T4`

Do not use TPU. Do not use CPU for the real training run.

In [ ]:
# Edit only this cell before running.
# If you pushed your code to your own fork/branch, set both values below.

REPO_URL = "https://github.com/ch33nchan/video_compress.git"
BRANCH = "main"

REPO_DIR = "/content/comma_video_compression_challenge"
DRIVE_OUT = "/content/drive/MyDrive/comma_video_runs/latent_pair_codec"

# Safe bring-up settings.
EPOCHS = 5
BATCH_SIZE = 2
DEVICE = "cuda"

# After the first successful run, try:
# EPOCHS = 10
# BATCH_SIZE = 4


## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. System packages

In [ ]:
!apt-get update -y
!apt-get install -y git-lfs ffmpeg
!git lfs install

## 3. Clone or refresh repo

In [ ]:
import os

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
    !git fetch --all --tags
    !git reset --hard
    !git clean -fd
    !git checkout {BRANCH}
    !git pull --ff-only
else:
    %cd /content
    !git clone {REPO_URL}
    %cd {REPO_DIR}
    !git checkout {BRANCH}

!git lfs pull
!pwd
!git remote -v
!git branch --show-current

## 4. Python environment

In [ ]:
%cd {REPO_DIR}
!python -m pip install --upgrade pip uv
!uv venv --python 3.11
!bash -lc 'source .venv/bin/activate && uv sync --group cu128'

## 5. Verify GPU and imports

You want to see `cuda_available True` and an NVIDIA GPU name.
If you see CPU-only, stop and reconnect to a GPU runtime.

In [ ]:
%cd {REPO_DIR}
!nvidia-smi || true
!bash -lc 'source .venv/bin/activate && python - <<"PY"
import torch, av, timm, einops, safetensors, segmentation_models_pytorch, brotli
print("torch", torch.__version__)
print("cuda_available", torch.cuda.is_available())
print("cuda_device_count", torch.cuda.device_count())
if torch.cuda.is_available():
    print("gpu", torch.cuda.get_device_name(0))
print("imports ok")
PY'

## 6. Train submission

In [ ]:
%cd {REPO_DIR}
!mkdir -p {DRIVE_OUT}
!bash -lc 'source .venv/bin/activate && bash submissions/latent_pair_codec/compress.sh --device {DEVICE} --epochs {EPOCHS} --batch-size {BATCH_SIZE}'

## 7. Evaluate submission

In [ ]:
%cd {REPO_DIR}
!bash -lc 'source .venv/bin/activate && bash evaluate.sh --submission-dir ./submissions/latent_pair_codec --device {DEVICE}'

## 8. Save artifacts to Drive

In [ ]:
%cd {REPO_DIR}
!mkdir -p {DRIVE_OUT}
!cp -f submissions/latent_pair_codec/archive.zip {DRIVE_OUT}/archive.zip
!cp -f submissions/latent_pair_codec/report.txt {DRIVE_OUT}/report.txt || true
!cp -f submissions/latent_pair_codec/archive/model.pt.br {DRIVE_OUT}/model.pt.br || true
!cp -f submissions/latent_pair_codec/archive/latents.pt.br {DRIVE_OUT}/latents.pt.br || true
!ls -lh {DRIVE_OUT}